In [1]:
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../../../datasets/processed/PAN2011_300")
SUSPICIOUS_DOC_ID = "part1__suspicious-document00007.txt"

# Step 1 - load everything
candidates_df = pd.read_parquet(PROCESSED_DIR / "embedding_candidates_suspicious.parquet")
top20_df = pd.read_parquet(Path("../../top20_df.parquet"))
source_chunks = pd.read_parquet(PROCESSED_DIR / "source_chunks.parquet")
suspicious_chunks = pd.read_parquet(PROCESSED_DIR / "suspicious_chunks_embeddings.parquet")

# Step 2 - filter candidates to top 20 source docs only
top_source_ids = set(top20_df["source_doc_id"].tolist())

candidates_filtered = candidates_df[
    (candidates_df["suspicious_doc_id"] == SUSPICIOUS_DOC_ID) &
    (candidates_df["source_doc_id"].isin(top_source_ids))
].copy()

print(f"Filtered candidates: {len(candidates_filtered)}")
print(candidates_filtered.columns.tolist())

Filtered candidates: 457
['suspicious_chunk_id', 'suspicious_doc_id', 'suspicious_chunk_index', 'suspicious_start_char', 'suspicious_end_char', 'source_chunk_id', 'source_doc_id', 'source_chunk_index', 'source_start_char', 'source_end_char', 'embedding_score', 'embedding_rank']


In [2]:
# Determine the text column name in each parquet (preprocessing may differ)
susp_text_col = "embedding_text"
src_text_col  = "chunk_text"

susp_text = (
    suspicious_chunks[suspicious_chunks["doc_id"] == SUSPICIOUS_DOC_ID]
    [["chunk_id", susp_text_col]]
    .rename(columns={"chunk_id": "suspicious_chunk_id", susp_text_col: "suspicious_text"})
)

src_text = (
    source_chunks[source_chunks["doc_id"].isin(top_source_ids)]
    [["chunk_id", src_text_col]]
    .rename(columns={"chunk_id": "source_chunk_id", src_text_col: "source_text"})
)

pairs_df = (
    candidates_filtered
    .merge(susp_text, on="suspicious_chunk_id", how="inner")
    .merge(src_text,  on="source_chunk_id",     how="inner")
)

# Keep top-10 highest-similarity pairs per source doc to feed the LLM
TOP_PAIRS_PER_DOC = 25
top_pairs = (
    pairs_df
    .sort_values("embedding_score", ascending=False)
    .groupby("source_doc_id")
    .head(TOP_PAIRS_PER_DOC)
    .reset_index(drop=True)
)

print(f"Source docs to evaluate: {top_pairs['source_doc_id'].nunique()}")
print(f"Total pairs sent to LLM: {len(top_pairs)}")
top_pairs[["source_doc_id", "embedding_score", "suspicious_text", "source_text"]].head()


Source docs to evaluate: 20
Total pairs sent to LLM: 254


,source_doc_id,embedding_score,suspicious_text,source_text
0,part13__source-document06022.txt,0.924300,"reason why you should accept my excuse, and he...","nor twice, I have not ventured persistently to..."
1,part13__source-document06022.txt,0.829731,"as i believed, of all painters whatsoever. And...","last of men to tell you so, had I trusted my o..."
2,part13__source-document06022.txt,0.807646,"and least, when removed some months from the e...","at the places where they exist, and cause a sl..."
3,part13__source-document06022.txt,0.784380,"reason why you should accept my excuse, and he...",must express themselves by art; and to say tha...
4,part13__source-document06022.txt,0.738564,meant finally for committee of Apollo archeget...,"if you could interpret that art rightly, the b..."


In [3]:
# ── Step 1: threshold filter ────────────────────────────────────────────────
import pandas as pd
LLM_SCORE_THRESHOLD = 0.95

llm_scores_df = pd.read_parquet("../05_text_alignment/llm_scores_df.parquet")
confirmed_sources = llm_scores_df[llm_scores_df["llm_score"] >= LLM_SCORE_THRESHOLD].copy()
print(f"Confirmed source docs (score >= {LLM_SCORE_THRESHOLD}): {len(confirmed_sources)}")
confirmed_sources[["source_doc_id", "llm_score", "llm_is_likely_source"]]


Confirmed source docs (score >= 0.95): 3


,source_doc_id,llm_score,llm_is_likely_source
0,part10__source-document04659.txt,0.95,True
1,part13__source-document06022.txt,0.95,True
2,part23__source-document11043.txt,0.95,True


In [4]:
# ── Step 2: classify plagiarism type per chunk pair ─────────────────────────
import time, json, re
from ollama import chat
from tqdm import tqdm

OLLAMA_MODEL = "gemma4:e4b"

PLAGIARISM_TYPES = ["copy_paste", "paraphrase", "shake", "none"]

def classify_chunk_pair(suspicious_text: str, source_text: str) -> dict:
    prompt = (
        "You are a plagiarism detection expert.\n\n"
        "Compare the SUSPICIOUS chunk and the SOURCE chunk below.\n\n"
        f"SUSPICIOUS:\n{suspicious_text}\n\n"
        f"SOURCE:\n{source_text}\n\n"
        "Classify the relationship into exactly one of these types:\n"
        "  - copy_paste : text is copied verbatim or near-verbatim (< 5% change)\n"
        "  - paraphrase : meaning preserved but sentences restructured or rewritten\n"
        "  - shake      : words replaced by synonyms / light edits, same structure\n"
        "  - none       : no meaningful plagiarism detected\n\n"
        "Also rate your confidence (0.0-1.0).\n"
        "Respond with ONLY a JSON object — no markdown — with keys: "
        "plagiarism_type (string), confidence (float 0-1), reasoning (string)."
    )

    response = chat(
        model=OLLAMA_MODEL,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
        think=False,
    )

    raw = response.message.content
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON in response: {raw[:300]}")
    data = json.loads(match.group())

    ptype = data.get("plagiarism_type", "none")
    if ptype not in PLAGIARISM_TYPES:
        ptype = "none"

    return {
        "plagiarism_type": ptype,
        "type_confidence": float(data.get("confidence", 0.0)),
        "type_reasoning": data.get("reasoning", ""),
    }


# Only run on pairs whose source doc passed the threshold
confirmed_source_ids = set(confirmed_sources["source_doc_id"].tolist())

confirmed_pairs = pairs_df[
    pairs_df["source_doc_id"].isin(confirmed_source_ids)
].copy()

print(f"Chunk pairs to classify: {len(confirmed_pairs)}")


Chunk pairs to classify: 212


In [5]:
# ── Step 3: run classification ───────────────────────────────────────────────
classification_rows = []

for row in tqdm(confirmed_pairs.itertuples(index=False), total=len(confirmed_pairs), desc="Classifying pairs"):
    try:
        result = classify_chunk_pair(row.suspicious_text, row.source_text)
    except Exception as e:
        result = {"plagiarism_type": "none", "type_confidence": 0.0, "type_reasoning": f"Error: {e}"}

    classification_rows.append({
        "suspicious_chunk_id":  row.suspicious_chunk_id,
        "suspicious_doc_id":    row.suspicious_doc_id,
        "suspicious_start_char": row.suspicious_start_char,
        "suspicious_end_char":  row.suspicious_end_char,
        "source_chunk_id":      row.source_chunk_id,
        "source_doc_id":        row.source_doc_id,
        "source_start_char":    row.source_start_char,
        "source_end_char":      row.source_end_char,
        "embedding_score":      row.embedding_score,
        "suspicious_text":      row.suspicious_text,
        "source_text":          row.source_text,
        **result,
    })

alignment_df = pd.DataFrame(classification_rows)
alignment_df = alignment_df.sort_values(
    ["source_doc_id", "embedding_score"], ascending=[True, False]
).reset_index(drop=True)

print(f"\nClassification summary:")
print(alignment_df["plagiarism_type"].value_counts().to_string())
alignment_df[["source_doc_id", "suspicious_start_char", "source_start_char", "embedding_score", "plagiarism_type", "type_confidence"]].head(10)


Classifying pairs: 100%|██████████| 212/212 [10:57<00:00,  3.10s/it]


Classification summary:
plagiarism_type
none          143
paraphrase     55
shake          10
copy_paste      4


,source_doc_id,suspicious_start_char,source_start_char,embedding_score,plagiarism_type,type_confidence
0,part10__source-document04659.txt,3766,230167,0.669662,paraphrase,0.9
1,part10__source-document04659.txt,3766,118146,0.595142,none,1.0
2,part10__source-document04659.txt,4974,230167,0.591690,none,1.0
3,part10__source-document04659.txt,4974,381625,0.589328,none,1.0
4,part10__source-document04659.txt,9901,174002,0.581333,none,1.0
5,part10__source-document04659.txt,3766,399049,0.578556,paraphrase,0.9
6,part10__source-document04659.txt,4974,399049,0.576803,none,0.9
7,part10__source-document04659.txt,4974,118146,0.572297,none,1.0
8,part10__source-document04659.txt,3766,397817,0.571886,paraphrase,0.9
9,part10__source-document04659.txt,3766,400400,0.566269,none,0.9


In [6]:
# ── Step 4: save span-level results (ready for PAN 2011 analytics) ──────────
OUTPUT_PATH = PROCESSED_DIR / "alignment_classified_spans.parquet"

alignment_df.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved {len(alignment_df)} classified spans to: {OUTPUT_PATH}")

# Quick breakdown per source doc + plagiarism type
summary = (
    alignment_df[alignment_df["plagiarism_type"] != "none"]
    .groupby(["source_doc_id", "plagiarism_type"])
    .agg(count=("plagiarism_type", "size"), mean_confidence=("type_confidence", "mean"))
    .reset_index()
    .sort_values(["mean_confidence", "count"], ascending=[False, False])
)
summary


Saved 212 classified spans to: ..\..\..\datasets\processed\PAN2011_300\alignment_classified_spans.parquet


,source_doc_id,plagiarism_type,count,mean_confidence
1,part13__source-document06022.txt,copy_paste,4,0.950000
3,part13__source-document06022.txt,shake,10,0.920000
2,part13__source-document06022.txt,paraphrase,44,0.885227
0,part10__source-document04659.txt,paraphrase,4,0.875000
4,part23__source-document11043.txt,paraphrase,7,0.871429


In [ ]:
# ── Step 5: keep highest-confidence NON-NONE span per suspicious chunk ────────
# Sort so non-none types always beat none (regardless of confidence), then by confidence
best_spans = (
    alignment_df
    .assign(is_plagiarism=(alignment_df["plagiarism_type"] != "none").astype(int))
    .sort_values(["is_plagiarism", "type_confidence"], ascending=[False, False])
    .drop_duplicates(subset=["suspicious_chunk_id"], keep="first")
    .drop(columns=["is_plagiarism"])
    .reset_index(drop=True)
)

detected = best_spans[best_spans["plagiarism_type"] != "none"].copy()

print(f"Total classified rows:          {len(alignment_df)}")
print(f"After dedup (1 per chunk):      {len(best_spans)}")
print(f"Detected as plagiarism:         {len(detected)}")
print()
detected[["suspicious_chunk_id", "suspicious_start_char", "suspicious_end_char",
          "source_doc_id", "source_start_char", "source_end_char",
          "plagiarism_type", "type_confidence"]]

In [8]:
# ── Step 6: compare detected spans vs XML ground truth ──────────────────────
GROUND_TRUTH_PATH = Path("../../../datasets/processed/PAN2011_ground_truth/pan2011_plagiarism_spans.parquet")

gt_df = pd.read_parquet(GROUND_TRUTH_PATH)
gt_doc = gt_df[gt_df["suspicious_doc_id"] == SUSPICIOUS_DOC_ID].copy()

print(f"XML ground truth spans for {SUSPICIOUS_DOC_ID}: {len(gt_doc)}")
print(f"Our detected spans (post-dedup, non-none):      {len(detected)}")
print()

# A detected chunk is a HIT if its suspicious char range overlaps any GT span
# from the same source doc (overlap = not (end <= gt_start or start >= gt_end))
def overlaps(det_start, det_end, gt_start, gt_end):
    return not (det_end <= gt_start or det_start >= gt_end)

hit_flags = []
matched_gt_indices = set()

for _, det in detected.iterrows():
    hit = False
    for gt_idx, gt in gt_doc.iterrows():
        if det["source_doc_id"] != gt["source_doc_id"]:
            continue
        if overlaps(det["suspicious_start_char"], det["suspicious_end_char"],
                    gt["suspicious_offset"], gt["suspicious_end"]):
            hit = True
            matched_gt_indices.add(gt_idx)
    hit_flags.append(hit)

detected = detected.copy()
detected["matched_gt"] = hit_flags

hits   = detected["matched_gt"].sum()
missed_gt = len(gt_doc) - len(matched_gt_indices)

print("=== Detection vs Ground Truth ===")
print(f"GT spans total:          {len(gt_doc)}")
print(f"GT spans matched:        {len(matched_gt_indices)}")
print(f"GT spans missed:         {missed_gt}")
print()
print(f"Detected spans:          {len(detected)}")
print(f"  True positives (hits): {hits}")
print(f"  False positives:       {len(detected) - hits}")
print()

precision = hits / len(detected) if len(detected) else 0
recall    = len(matched_gt_indices) / len(gt_doc) if len(gt_doc) else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1:        {f1:.3f}")
print()

# Per-type breakdown of hits vs misses
print("=== Hit rate by plagiarism type ===")
print(detected.groupby("plagiarism_type")["matched_gt"].value_counts().unstack(fill_value=0).to_string())


XML ground truth spans for part1__suspicious-document00007.txt: 6
Our detected spans (post-dedup, non-none):      1

=== Detection vs Ground Truth ===
GT spans total:          6
GT spans matched:        0
GT spans missed:         6

Detected spans:          1
  True positives (hits): 0
  False positives:       1

Precision: 0.000
Recall:    0.000
F1:        0.000

=== Hit rate by plagiarism type ===
matched_gt       False
plagiarism_type       
shake                1
